### I have 3 solutions for this problem.

In all solutions I have 3 bots:
1. George - an argumentative bot. Using gpt 5 mini for this.
2. Graham - a polite, courteous bot. Using gemini-3.1-flash-lite or gpt 5 nano for this
3. Ollie - a chaotic bot who speaks in riddles and is a mediator between the other 2 bots. Using ollama gpt-oss:20b for this

The first solution is an extension of the 2-bot conversation.

The second solution passes the entire conversation to the bots. This particular solution requires 3 different call functions at present, but it can be changed to just 1 common function by using LiteLLM and then passing the model name as a variable.

The third solution is a variant of the second one - just that I have used LiteLLM , which means that the call api function has been reduced from 3 to 1, making it a very efficient solution.

## Imports

In [ ]:
import os
from openai import OpenAI
from IPython.display import display,Markdown
from dotenv import load_dotenv
from litellm import completion

import requests

## Environment Setup

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API key exists and begins {openai_api_key[:8]}")
else:
    print(f"OpenAI API key not found")

if google_api_key:
    print(f"Gemini API key exists and begins {google_api_key[:8]}")
else:
    print(f"Gemini API key not found")

## Initiate LLMs

In [ ]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(base_url=gemini_url, api_key=google_api_key)

In [ ]:
requests.get("http://localhost:11434").content

In [ ]:
ollama = OpenAI(base_url=ollama_url,api_key='ollama')

In [ ]:
george_model = "gpt-5-mini"
graham_model = "gemini-3.1-flash-lite"
#graham_model = "gpt-5-nano"
ollie_model = "llama3.2"

## System Prompts

In [ ]:
# GPT 5 nano = George, GPT 5 mini = Graham, Ollama = Ollie

george_system_prompt = """
You are George, a chatbot who is very argumentative.
You disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Graham and Ollie.
"""

graham_system_prompt = """
You are Graham, you are a very polite, courteous chatbot. You try to agree with everything the other person says,
or find common ground. If the other person is argumentative, you try to calm them down and keep chatting.
You are in a conversation with George and Ollie.
"""

ollie_system_prompt = """
    Your name is Ollie. You are a chaotic chatbot.
    You speak in short, philosophical and slightly confusing riddles.
    You act as the wildcard mediator between George and Graham.
"""

## Solution 1

This simply extends the 2-chatbot solution

In [ ]:
def chat_george():
    messages = [{"role":"system","content":george_system_prompt}]
    for george,graham,ollie in zip(george_messages,graham_messages,ollie_messages):
        messages.append({"role":"assistant","content":george})
        messages.append({"role":"user","content":f"Graham: {graham}\nOllie: {ollie}"})
    response = openai.chat.completions.create(model=george_model,messages=messages)
    return response.choices[0].message.content    

In [ ]:
def chat_graham():
    messages = [{"role":"system","content":graham_system_prompt}]
    for george,graham,ollie in zip(george_messages,graham_messages,ollie_messages):
        messages.append({"role":"user","content":f"George: {george}\nOllie: {ollie}"})
        messages.append({"role":"assistant","content":graham})
    messages.append({"role":"user","content":f"George: {george_messages[-1]}"})
    response = gemini.chat.completions.create(model=graham_model,messages=messages)
    #response = openai.chat.completions.create(model=graham_model,messages=messages)
    return response.choices[0].message.content
    

In [ ]:
def chat_ollie():
    messages = [{"role":"system","content":ollie_system_prompt}]
    for george,graham,ollie in zip(george_messages,graham_messages,ollie_messages):
        messages.append({"role":"user","content":f"George: {george}\nGraham: {graham}"})
        messages.append({"role":"assistant","content":ollie})
    messages.append({"role":"user","content":f"George: {george_messages[-1]}\nGraham: {graham_messages[-1]}"})
    response = ollama.chat.completions.create(model=ollie_model,messages=messages)
    return response.choices[0].message.content

In [ ]:
george_messages = ["Hi there"]
graham_messages = ["Hi George, glad to be here!"]
ollie_messages = ["Greetings, travellers of thought!"]

display(Markdown(f"## Round 0"))
display(Markdown(f"### George:\n{george_messages[0]}\n"))
display(Markdown(f"### Graham:\n{graham_messages[0]}\n"))
display(Markdown(f"### Ollie:\n{ollie_messages[0]}\n"))


for i in range(2):
    print("-" * 75)
    display(Markdown(f"## Round {i+1}\n"))
    
    george_next = chat_george()
    display(Markdown(f"### George:\n{george_next}\n"))
    george_messages.append(george_next)    
    
    graham_next = chat_graham()
    display(Markdown(f"### Graham:\n{graham_next}\n"))
    graham_messages.append(graham_next)
    
    ollie_next = chat_ollie()
    display(Markdown(f"### Ollie:\n{ollie_next}\n"))
    ollie_messages.append(ollie_next)

## Solution 2

This solution uses coversation logic. It eliminates the "assistant" role and uses just the system prompt and user/conversation prompt

In [ ]:
def call_george(u,s,m):
    messages = [
        {"role":"system","content":s},
        {"role":"user","content":u}
    ]
    response = openai.chat.completions.create(model=m,messages=messages)
    return response.choices[0].message.content

In [ ]:
def call_graham(u,s,m):
    messages = [
        {"role":"system","content":s},
        {"role":"user","content":u}
    ]
    response = gemini.chat.completions.create(model=m,messages=messages)
    return response.choices[0].message.content

In [ ]:
def call_ollie(u,s,m):
    messages = [
        {"role":"system","content":s},
        {"role":"user","content":u}
    ]
    response = ollama.chat.completions.create(model=m,messages=messages)
    return response.choices[0].message.content

In [ ]:
def llm_user_prompt(llm,c,u1,u2):
    user_prompt = f"""
        You are {llm}, in conversation with {u1} and {u2}.
        The conversation so far is as follows:
        {c}
        Now with this, respond with what you would like to say next, as {llm}.
        """
    return user_prompt

In [ ]:
george_messages = "Hi there"
graham_messages = "Hi George, glad to be here!"
ollie_messages = "Greetings, travellers of thought!"

conversation = ""

conversation += f"George: {george_messages}\n"
conversation += f"Graham: {graham_messages}\n"
conversation += f"Ollie: {ollie_messages}\n"

display(Markdown(f"## Round 0"))
print(f"Conversation thus far:\n{conversation}\n")
print("-" * 75)

for i in range(2):

    display(Markdown(f"## Round {i+1}\n"))

    george_user_msg = llm_user_prompt("George",conversation,"Graham","Ollie")
    george_next=call_george(george_user_msg,george_system_prompt,george_model)
    display(Markdown(f"### George:\n{george_next}\n"))
    conversation += f"George: {george_next}\n"

    graham_user_msg = llm_user_prompt("Graham",conversation,"George","Ollie")
    graham_next=call_graham(graham_user_msg,graham_system_prompt,graham_model)
    display(Markdown(f"### Graham:\n{graham_next}\n"))
    conversation += f"Graham: {graham_next}\n"

    ollie_user_msg = llm_user_prompt("Ollie",conversation,"Graham","George")
    ollie_next=call_ollie(ollie_user_msg,ollie_system_prompt,ollie_model)
    display(Markdown(f"### Ollie:\n{ollie_next}\n"))
    conversation += f"Ollie: {ollie_next}\n"

    print("-" * 100)
    print(f"Conversation thus far:\n{conversation}\n")
    print("-" * 100)

## Solution 2a: Let's try LiteLLM

In [ ]:
def call_llm(u,s,m):
    messages = [
        {"role":"system","content":s},
        {"role":"user","content":u}
    ]
    if m.startswith("ollama/"):
        response = completion(model=m,messages=messages,api_base="http://localhost:11434")
    else:
        response = completion(model=m,messages=messages)
        print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
    print("-" * 100)
    print(f"Input tokens: {response.usage.prompt_tokens}")
    print(f"Output tokens: {response.usage.completion_tokens}")    
    print(f"Total tokens: {response.usage.total_tokens}")
    print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")
    print("-" * 100)
    return response.choices[0].message.content

In [ ]:
george_messages = "Hi there"
graham_messages = "Hi George, glad to be here!"
ollie_messages = "Greetings, travellers of thought!"

conversation = ""

conversation += f"George: {george_messages}\n"
conversation += f"Graham: {graham_messages}\n"
conversation += f"Ollie: {ollie_messages}\n"

display(Markdown(f"## Round 0"))
print(f"Conversation thus far:\n{conversation}\n")
print("-" * 100)

for i in range(2):

    display(Markdown(f"## Round {i+1}\n"))

    george_user_msg = llm_user_prompt("George",conversation,"Graham","Ollie")
    george_next=call_llm(george_user_msg,george_system_prompt,"openai/gpt-5-nano")
    display(Markdown(f"### George:\n{george_next}\n"))
    conversation += f"George: {george_next}\n"

    graham_user_msg = llm_user_prompt("Graham",conversation,"George","Ollie")
    graham_next=call_llm(graham_user_msg,graham_system_prompt,"gemini/gemini-3.1-flash-lite")
    display(Markdown(f"### Graham:\n{graham_next}\n"))
    conversation += f"Graham: {graham_next}\n"

    ollie_user_msg = llm_user_prompt("Ollie",conversation,"Graham","George")
    ollie_next=call_llm(ollie_user_msg,ollie_system_prompt,"ollama/llama3.2")
    display(Markdown(f"### Ollie:\n{ollie_next}\n"))
    conversation += f"Ollie: {ollie_next}\n"

    print("-" * 100)
    print(f"Conversation thus far:\n{conversation}\n")
    print("-" * 100)